# mini-vLLM — GPU checks and benchmark (Colab T4)

One run produces, in order: **a** fp16 agreement with HF, **b** memory pressure, **c** KV sizing, **d** the identical-outputs gate in float64 on the real model, **e** a prefix-caching benchmark on a shared-prefix workload, **e2** the admission-thrash simulation, **f** the main sweep. Results go to one zip with a `config.json`, one JSON file per check, and a `README.md`.

Settings live in the **Run configuration** cell. Nothing else needs editing. Colab clones `master` from GitHub, so **commit and push first**.

P26 (vLLM comparison, cost-model fit) waits until you have looked at the output of **f**.


In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("mini-vllm").exists():
        subprocess.run(["git", "clone", "--branch", "master", "https://github.com/AustinD123/mini-vllm.git"], check=True)
    os.chdir("mini-vllm")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "matplotlib", "pytest"], check=True)
else:
    # Local run: the dry run executes this notebook from its colab/ folder.
    here = Path.cwd()
    os.chdir(here.parent if here.name == "colab" else here)
sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd(), "| Colab:", IN_COLAB)

## Run configuration (T4 defaults)

In [ ]:
import json
import os
from pathlib import Path

DRY_RUN = os.environ.get("MINIVLLM_DRY_RUN") == "1"  # set by the local smoke test; leave unset on Colab

CONFIG = dict(
    # model and numerics
    model_id="Qwen/Qwen2.5-0.5B-Instruct",
    dtype="float16",
    kv_memory_fraction=0.5,      # share of free GPU memory given to the KV pool
    repo_ref="master",           # Colab clones this branch; commit and push before running
    seed=0,
    # main sweep (no shared prefix)
    # requests per second; every point is repeated. 0.25 is dropped to fit the 60-minute budget: with all five
    # rates and 3 repeats the estimate is 72 min (sequential above 0.25 skipped). Restoring it needs a budget change.
    rates=[0.5, 1.0, 2.0, 4.0],
    sequential_max_rate=0.5,            # sequential is saturated above this rate, so it is skipped there
    repeats=3,                          # wall-clock repeats per point: median and min-max reported
    num_requests=48,
    systems=("sequential", "static", "engine"),
    max_batch=16,
    token_budget=512,
    # workload lengths: ShareGPT-fitted lognormal defaults, clipped at max_len
    max_len=256,
    prompt_mu=3.835046, prompt_sigma=1.750131,
    output_mu=5.183535, output_sigma=1.160675,
    # prefix-caching benchmark (shared system prompt, Poisson arrivals)
    prefix_rates=[2.0, 4.0],
    prefix_num_requests=48,
    prefix_shared_len=128,
    # admission thrash: engine preemptions against KV pool size (blocks)
    preemption_rate=8.0,
    preemption_pool_sizes=[32, 64, 128, 256],
    # agreement checks
    agreement_requests=16,
    # output
    save_to_drive=False,         # True: write results straight to Google Drive as they finish
)
if DRY_RUN:  # a few requests and two rates, to check the whole notebook runs
    CONFIG.update(rates=[0.25, 2.0], num_requests=6, prefix_rates=[4.0], prefix_num_requests=6,
                  prefix_shared_len=32, agreement_requests=4, max_len=64,
                  preemption_pool_sizes=[8, 16])

# Assumed seconds per model step on a T4. Used only by the runtime estimate; not measured yet.
# HF forward: about 34 ms on a T4 (owner's measurement). The engine step at batch 32 is about 1.7x an
# HF forward on the 3050 (docs/perf-diagnosis.md, section 6), so about 60 ms is assumed for the engine.
# Replace both with the first run's numbers.
STEP_S = {"sequential": 0.034, "static": 0.034, "engine": 0.060, "engine_prefix": 0.060}
SETUP_MIN = 6       # model loads, agreement checks, gate and sizing (rough)


def estimate_minutes(c):
    from minivllm.bench.workload import make_workload
    lengths = dict(prompt_mu=c["prompt_mu"], prompt_sigma=c["prompt_sigma"], output_mu=c["output_mu"],
                   output_sigma=c["output_sigma"], max_len=c["max_len"])
    b = c["max_batch"]

    def steps(system, rate, num, shared=0):
        reqs = make_workload(rate=rate, num_requests=num, seed=c["seed"], shared_prefix_len=shared, **lengths)
        outs = [r.output_len for r in reqs]
        if system == "sequential":
            n = sum(outs) + num                      # one prefill and one forward per output token
        elif system == "static":
            n = sum(max(outs[i:i + b]) + 1 for i in range(0, num, b))
        else:
            n = 1.5 * sum(outs) / b + num / b        # continuous batching: about 1.5x the ideal step count
        return n * STEP_S[system], reqs[-1].arrival_time

    main = 0.0
    for rate in c["rates"]:
        for system in c["systems"]:
            if system == "sequential" and rate > c["sequential_max_rate"]:
                continue
            t, span = steps(system, rate, c["num_requests"])
            main += max(t, span)                     # arrivals overlap compute; the longer one sets the time
    prefix = 0.0
    for rate in c["prefix_rates"]:
        for system in ("engine", "engine_prefix"):
            t, span = steps(system, rate, c["prefix_num_requests"], c["prefix_shared_len"])
            prefix += max(t, span)
    pre_t, _ = steps("engine", c["preemption_rate"], c["num_requests"])
    pre = len(c["preemption_pool_sizes"]) * pre_t    # simulated clock, but the forwards run on the GPU
    return (c["repeats"] * (main + prefix) + pre) / 60 + SETUP_MIN


ESTIMATE_MIN = estimate_minutes(CONFIG)
print(f"estimated T4 runtime: {ESTIMATE_MIN:.0f} min (assumes {STEP_S} s per step; not yet measured)")
if ESTIMATE_MIN > 60:
    print("WARNING: over the 60-minute budget. Drop a rate from CONFIG['rates'] or CONFIG['prefix_rates'] (not the repeats).")

if CONFIG["save_to_drive"]:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/mini-vllm/results")
else:
    OUT_DIR = Path("results") / ("dry-run" if DRY_RUN else "t4")
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "config.json").write_text(json.dumps({**CONFIG, "estimate_min": ESTIMATE_MIN, "step_s": STEP_S},
                                                indent=2, default=list), encoding="utf-8")
print("DRY_RUN:", DRY_RUN, "| output:", OUT_DIR.resolve())
for key, value in CONFIG.items():
    print(f"  {key}: {value}")


## Load the model twice: HF attention (reference) and the paged backend (engine)

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from minivllm.attention.backend import BACKEND_NAME, register_backend

assert torch.cuda.is_available(), "switch the runtime to a GPU (Runtime > Change runtime type)"
DTYPE = getattr(torch, CONFIG["dtype"])
DEVICE = "cuda"
register_backend()
ref = AutoModelForCausalLM.from_pretrained(CONFIG["model_id"], dtype=DTYPE, attn_implementation="sdpa").eval().to(DEVICE)
paged = AutoModelForCausalLM.from_pretrained(CONFIG["model_id"], dtype=DTYPE, attn_implementation=BACKEND_NAME).eval().to(DEVICE)
cfg = ref.config
print(torch.cuda.get_device_name(0), "| torch", torch.__version__, "| free MiB", round(torch.cuda.mem_get_info()[0] / 2**20))

## Seeded workload (the same generator as the sweep)

In [ ]:
from minivllm.bench.workload import make_workload
from minivllm.bench.jobs import build_jobs
from minivllm.bench.sweep import run_sweep
from minivllm.bench.plots import plot_sweep, plot_prefix_sweep, sweep_table
from minivllm.bench.results import validate_result
from minivllm.clock import WallClock
from minivllm.bench.repeats import aggregate_repeats
from minivllm.bench.results import write_result
from pathlib import Path


def repeat_sweep(out_base, **kwargs):
    # Run one sweep CONFIG[repeats] times on wall-clock timing; return one median record per point.
    out_base = Path(out_base)
    runs = [run_sweep(ref, paged, out_dir=out_base / f"repeat{k}", clock_factory=WallClock, **kwargs)
            for k in range(CONFIG["repeats"])]
    aggregated = aggregate_repeats(runs)
    for r in aggregated:
        validate_result(r)
        write_result(out_base / "aggregated" / f"{r['system']}_rate{r['workload']['rate']:g}.json", r)
    return aggregated


def span(m, s, name, scale=1.0, digits=3, unit=""):
    # Median, then [min-max] over repeats, for one metric.
    lo, hi = s[name]
    def fmt(v):
        return f"{v * scale:.{digits}f}"
    return f"{fmt(m[name])}{unit} [{fmt(lo)}-{fmt(hi)}{unit}]"


def show_medians(records):
    for r in records:
        m, s = r["metrics"], r["spread"]
        print(f"{r['system']:>14} rate {r['workload']['rate']:g} (median of {r['repeats']}):")
        print("    TTFT p50", span(m, s, "ttft_50", unit=" s"), "| TTFT p99", span(m, s, "ttft_99", unit=" s"))
        print("    TPOT p50", span(m, s, "tpot_50", scale=1000, digits=1, unit=" ms"),
              "| throughput", span(m, s, "throughput", digits=1, unit=" tok/s"))
        if m["kv_in_use_peak_fraction"] is not None:  # the baselines have no KV pool
            print(f"    prefix hit rate {m['prefix_hit_rate']:.1%} | KV in use (peak) {m['kv_in_use_peak_fraction']:.1%}"
                  f" | pool occupancy (peak) {m['kv_occupancy_peak_fraction']:.1%}")

LENGTHS = dict(prompt_mu=CONFIG["prompt_mu"], prompt_sigma=CONFIG["prompt_sigma"],
               output_mu=CONFIG["output_mu"], output_sigma=CONFIG["output_sigma"], max_len=CONFIG["max_len"])
requests = make_workload(rate=10.0, num_requests=CONFIG["agreement_requests"], seed=CONFIG["seed"], **LENGTHS)
jobs = build_jobs(requests, vocab_size=cfg.vocab_size, seed=CONFIG["seed"])
prompts = [list(j.prompt_ids) for j in jobs]
lens = [j.output_len for j in jobs]
print(len(jobs), "requests; prompt lengths", [len(p) for p in prompts][:8], "...")

## a. fp16 engine vs HF generate, same dtype. Headline: free-running agreement and first divergence. Teacher-forced is the per-step check.

In [ ]:
from minivllm.bench.gpu_checks import agreement, run_engine, teacher_forced_agreement

ref_out = {}
for j in jobs:
    ids = torch.tensor([list(j.prompt_ids)], device=DEVICE)
    with torch.no_grad():
        out = ref.generate(ids, do_sample=False, max_new_tokens=j.output_len, min_new_tokens=j.output_len,
                           eos_token_id=None, pad_token_id=0, repetition_penalty=1.0)
    ref_out[j.request_id] = out[0, len(j.prompt_ids):].tolist()

engine_out, preemptions = run_engine(paged, prompts, lens, num_blocks=512, max_batch=CONFIG["max_batch"],
                                     token_budget=CONFIG["token_budget"])
a = agreement(engine_out, ref_out)
t = teacher_forced_agreement(paged, prompts, ref_out)
print(f"free-running top-1: {a['top1']:.4%} over {a['positions']} positions")
print("first divergence per request:", a["first_divergence"])
print(f"teacher-forced top-1: {t['top1']:.4%} over {t['positions']} positions (target > 99%)")
print("preemptions:", preemptions)
(OUT_DIR / "agreement.json").write_text(json.dumps({
    "model": CONFIG["model_id"], "dtype": CONFIG["dtype"], "max_len": CONFIG["max_len"],
    "requests": len(jobs), "free_running": a, "teacher_forced": t, "preemptions": preemptions,
}, indent=2, default=str), encoding="utf-8")


## b. memory pressure: a shrunken pool must preempt, still finish, and still agree

In [ ]:
import math
need = [math.ceil((len(p) + n) / 16) for p, n in zip(prompts, lens)]
small_pool = max(need) + 2  # enough for the largest request alone, not for the whole batch
pressed_out, pressed_preemptions = run_engine(paged, prompts, lens, num_blocks=small_pool,
                                              max_batch=CONFIG["max_batch"], token_budget=CONFIG["token_budget"])
finished = all(len(pressed_out[j.request_id]) == j.output_len for j in jobs)
b = agreement(pressed_out, ref_out)
print("pool blocks:", small_pool, "| preemptions:", pressed_preemptions)
print("all requests finished:", finished)
print(f"free-running top-1 under pressure: {b['top1']:.4%}")
(OUT_DIR / "agreement_pressure.json").write_text(json.dumps({
    "pool_blocks": small_pool, "preemptions": pressed_preemptions, "all_finished": finished,
    "free_running": b,
}, indent=2, default=str), encoding="utf-8")
assert finished, "some request did not finish under memory pressure"


## c. sizing: allocate the P24-predicted blocks; compare measured peak with the prediction

In [ ]:
from minivllm.bench.gpu_checks import sizing_check

sizing = sizing_check(cfg, DTYPE, fraction=CONFIG["kv_memory_fraction"])
for key, value in sizing.items():
    print(f"{key:>22}: {value}")
(OUT_DIR / "sizing.json").write_text(json.dumps(sizing, indent=2, default=str), encoding="utf-8")


## d. the identical-outputs gate, float64 on the real model (sequential HF, static HF, engine batched, engine per-sequence)

Float64 is the exactness setting the test suite uses. In fp16, batching changes the numerics, so bitwise identity is not expected there. The dry run gates a tiny float64 model instead, because the local GPU has 4 GB. The T4 run gates the real model: two float64 copies take about 8 GB.


In [ ]:
from minivllm.bench.gpu_checks import identical_outputs_gate
from minivllm.bench.jobs import build_jobs
from minivllm.attention.backend import BACKEND_NAME
from transformers import Qwen2Config, Qwen2ForCausalLM

if DRY_RUN:
    tiny = dict(vocab_size=50, hidden_size=32, intermediate_size=64, num_hidden_layers=2, num_attention_heads=4,
                num_key_value_heads=2, max_position_embeddings=512, eos_token_id=None, bos_token_id=None)
    torch.manual_seed(CONFIG["seed"])
    gate_ref = Qwen2ForCausalLM(Qwen2Config(**tiny, attn_implementation="sdpa")).eval().double().to(DEVICE)
    gate_paged = Qwen2ForCausalLM(Qwen2Config(**tiny, attn_implementation=BACKEND_NAME)).eval().double().to(DEVICE)
    gate_paged.load_state_dict(gate_ref.state_dict())
    gate_jobs = build_jobs(requests, vocab_size=50, seed=CONFIG["seed"])
    gate_model = "tiny float64 (dry run)"
else:
    gate_ref = AutoModelForCausalLM.from_pretrained(CONFIG["model_id"], dtype=torch.float64, attn_implementation="sdpa").eval().to(DEVICE)
    gate_paged = AutoModelForCausalLM.from_pretrained(CONFIG["model_id"], dtype=torch.float64, attn_implementation=BACKEND_NAME).eval().to(DEVICE)
    gate_jobs = jobs
    gate_model = f"{CONFIG['model_id']} (float64)"

gate = identical_outputs_gate(gate_ref, gate_paged, gate_jobs, max_batch=CONFIG["max_batch"],
                              token_budget=CONFIG["token_budget"], num_blocks=512)
gate["model"] = gate_model
(OUT_DIR / "identical_outputs_gate.json").write_text(json.dumps(gate, indent=2, default=str), encoding="utf-8")
print("model:", gate_model, "| requests:", gate["requests"], "| output tokens:", gate["output_tokens"])
for name, ok in gate["checks"].items():
    print(f"  {name}: {ok}")
print("first mismatch:", gate["first_mismatch_request"])
del gate_ref, gate_paged
torch.cuda.empty_cache()
assert gate["passed"], "identical-outputs gate failed"


In [ ]:
!python -m pytest -q tests/test_baselines.py -k gate --junitxml={(OUT_DIR / "gate_pytest.xml").resolve()}


## e. prefix caching on a shared-prefix workload (Poisson arrivals). Engine vs engine with PrefixCache.

In [ ]:
PREFIX_LENGTHS = dict(LENGTHS, shared_prefix_len=CONFIG["prefix_shared_len"])
prefix_records = repeat_sweep(
    OUT_DIR / "prefix",
    rates=CONFIG["prefix_rates"],
    num_requests=CONFIG["prefix_num_requests"],
    seed=CONFIG["seed"],
    systems=("engine", "engine_prefix"),
    vocab_size=cfg.vocab_size,
    max_batch=CONFIG["max_batch"],
    token_budget=CONFIG["token_budget"],
    kv_memory_fraction=CONFIG["kv_memory_fraction"],
    length_overrides=PREFIX_LENGTHS,
)
show_medians(prefix_records)
plot_prefix_sweep(prefix_records, OUT_DIR / "prefix" / "plots")

## e2. admission thrash (simulated): preemptions against KV pool size, cost-model clock, engine only

In [ ]:
from minivllm.bench.sweep import run_preemption_sweep
from minivllm.bench.plots import plot_preemptions

pre_records = run_preemption_sweep(
    ref, paged,
    rate=CONFIG["preemption_rate"],
    num_requests=CONFIG["num_requests"],
    seed=CONFIG["seed"],
    out_dir=OUT_DIR / "preemption",
    pool_sizes=CONFIG["preemption_pool_sizes"],
    max_batch=CONFIG["max_batch"],
    token_budget=CONFIG["token_budget"],
    length_overrides=LENGTHS,
)
for r in pre_records:
    validate_result(r)
    print(f"pool {r['workload']['num_blocks']:>4} blocks: preemptions {r['metrics']['num_preemptions']}, "
          f"KV in use (peak) {r['metrics']['kv_in_use_peak_fraction']:.1%}")
plot_preemptions(pre_records, OUT_DIR / "preemption" / "plots")

## f. main sweep: sequential, static and engine across request rates

In [ ]:
seq_rates = [r for r in CONFIG["rates"] if r <= CONFIG["sequential_max_rate"]]
engine_rates = [r for r in CONFIG["rates"] if r > CONFIG["sequential_max_rate"]]
records = repeat_sweep(
    OUT_DIR / "main_seq",
    rates=seq_rates,
    num_requests=CONFIG["num_requests"],
    seed=CONFIG["seed"],
    systems=CONFIG["systems"],
    vocab_size=cfg.vocab_size,
    max_batch=CONFIG["max_batch"],
    token_budget=CONFIG["token_budget"],
    kv_memory_fraction=CONFIG["kv_memory_fraction"],
    length_overrides=LENGTHS,
)
if engine_rates:  # above sequential_max_rate sequential is saturated, so only static and engine run there
    records += repeat_sweep(
        OUT_DIR / "main_batched",
        rates=engine_rates,
        num_requests=CONFIG["num_requests"],
        seed=CONFIG["seed"],
        systems=tuple(s for s in CONFIG["systems"] if s != "sequential"),
        vocab_size=cfg.vocab_size,
        max_batch=CONFIG["max_batch"],
        token_budget=CONFIG["token_budget"],
        kv_memory_fraction=CONFIG["kv_memory_fraction"],
        length_overrides=LENGTHS,
    )
records.sort(key=lambda r: (r["workload"]["rate"], r["system"]))
show_medians(records)
for row in sweep_table(records):
    print(row)
(OUT_DIR / "main").mkdir(parents=True, exist_ok=True)
plot_sweep(records, OUT_DIR / "main" / "plots")


In [ ]:
from IPython.display import Image, display
import csv

rows = sweep_table(records)
with open(OUT_DIR / "main" / "table.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
for path in (sorted((OUT_DIR / "main" / "plots").glob("*.png")) + sorted((OUT_DIR / "prefix" / "plots").glob("*.png"))
             + sorted((OUT_DIR / "preemption" / "plots").glob("*.png"))):
    display(Image(str(path)))


## Write the README and package the results


In [ ]:
import shutil, subprocess, time

commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip() or "unknown"
gpu = torch.cuda.get_device_name(0)
(OUT_DIR / "README.md").write_text(f"""# mini-vLLM results ({'dry run' if DRY_RUN else 'T4'})

- Commit: `{commit}`
- GPU: {gpu} | torch {torch.__version__} | model {CONFIG['model_id']} ({CONFIG['dtype']} serving; the gate is float64)
- **max_len: {CONFIG['max_len']}** tokens (workload prompt plus output, clipped to this)
- Seed {CONFIG['seed']} | wall-clock timing, {CONFIG['repeats']} repeats per point (median, min-max)
- Main sweep rates: {CONFIG['rates']} req/s. Sequential runs only up to {CONFIG['sequential_max_rate']} req/s.
- Runtime estimate before the run: {ESTIMATE_MIN:.0f} min, from the assumed step times in `config.json` (not measured)

## Files

- `config.json`: every setting used in this run
- `agreement.json`, `agreement_pressure.json`: fp16 agreement with HF `generate` (free-running, first divergence, teacher-forced), and the same under a shrunken pool
- `sizing.json`: KV pool bytes predicted from free memory against bytes measured on the GPU
- `identical_outputs_gate.json`: sequential, static and engine (batched and per-sequence attention) give the same greedy tokens, in float64
- `gate_pytest.xml`: the same gate from the test suite, in JUnit form
- `main/`: main sweep table and plots; `main_seq/`, `main_batched/`: the raw per-repeat records
- `prefix/`, `preemption/`: prefix-caching and admission-thrash results, with plots
""", encoding="utf-8")
stamp = time.strftime("%Y%m%d-%H%M%S")
zip_base = Path(f"mini-vllm-results-{'dry-run' if DRY_RUN else 't4'}-{stamp}")
archive = shutil.make_archive(str(zip_base), "zip", root_dir=OUT_DIR)
print("wrote", archive, "| size MiB", round(Path(archive).stat().st_size / 2**20, 2))
if IN_COLAB and not CONFIG["save_to_drive"]:
    from google.colab import files
    files.download(archive)
